# Session 3: Can It Survive Regime Shifts? HMM Backtesting and Zero-Level AI Baselines

We have a rebalancing engine. It adapts to sentiment, enforces safety rules, and outperforms static portfolios on smooth synthetic data. But the real question is: _can it survive a regime shift?_ In this session, we use Hidden Markov Models with Poisson jumps ([JumpHMM.jl](https://github.com/varnerlab/JumpHMM.jl)) to generate synthetic market paths that include normal markets, stressed markets, and sudden crashes — then we run the engine across hundreds of these paths to see if it holds up.

We also introduce a "zero-level AI baseline": what would happen if you just asked a language model for allocation advice? This gives us a lower bound on what counts as "AI-driven" portfolio management.

> **By the end of this session, you will be able to:**
> * Explain how Hidden Markov Models with Poisson jumps generate realistic regime-switching synthetic data
> * Fit and tune a JumpHMM model from price data and generate Monte Carlo test paths
> * Backtest the Session 2 rebalancing engine across hundreds of HMM-generated scenarios
> * Compare the engine to a naive "AI baseline" and produce a formal validation report with pass/fail criteria

Let's stress-test this engine for real.

## Examples
The following example notebooks accompany this lecture:

➡ [Example: HMM-Powered Backtesting](eCornell-AI-Finance-L3-Example-HMMBacktesting-May-2026.ipynb). In this example, we fit a JumpHMM model to synthetic training data, generate regime-switching test paths, and backtest the rebalancing engine across hundreds of Monte Carlo scenarios. We visualize the distribution of outcomes and identify failure modes.

➡ [Example: AI Baseline and Validation Report](eCornell-AI-Finance-L3-Example-AIBaselineValidation-May-2026.ipynb). In this example, we implement a "zero-level AI baseline" (naive momentum rebalancing), compare it to the engine and buy-and-hold, and produce a formal validation report with pass/fail criteria for deployment readiness.

## Why GBM Isn't Enough: The Case for Regime-Switching Models

In Sessions 1 and 2, we generated synthetic data using geometric Brownian motion (GBM) — returns drawn from a single multivariate normal distribution with constant drift and volatility. This is fine for prototyping, but it misses three features of real markets that matter most for stress testing:

> **1. Volatility clustering.** Real markets exhibit periods of high volatility followed by more high volatility (and vice versa). GBM assumes constant volatility — every day looks the same statistically.
>
> **2. Fat tails.** Real returns have more extreme events than a normal distribution predicts. The 2008 crash, the March 2020 COVID drop, and the April 2025 tariff shock were all multi-sigma events under GBM assumptions.
>
> **3. Regime persistence.** Markets don't randomly switch between calm and crisis — they tend to _stay_ in a regime for extended periods. A bear market lasts months, not minutes. GBM has no concept of "regimes."

Hidden Markov Models address all three: they model the market as switching between hidden states (regimes), each with its own return distribution, with persistent transitions between states.

## The JumpHMM: Hidden Markov Models with Poisson Jumps

The [JumpHMM.jl](https://github.com/varnerlab/JumpHMM.jl) package implements a hybrid Hidden Markov Model with a Poisson jump mechanism, following [Alswaidan and Varner (2025)](https://arxiv.org/abs/2603.10202). The model has four components:

> **1. State Space.** The continuous space of excess log-growth rates $G_t = \frac{1}{\Delta t}\ln\frac{P_t}{P_{t-1}} - r_f$ is discretized into $N$ states using equal-probability quantile bins of a fitted Laplace distribution.

> **2. Transition Matrix.** An $N \times N$ row-stochastic matrix $\mathbf{T}$ governs the probability of moving between states: $P(S_t = j \mid S_{t-1} = i) = T_{ij}$. This captures regime persistence — the diagonal elements represent the probability of _staying_ in the current state.

> **3. Emission Distributions.** Each state $k$ has its own Student-t emission: $G_t \mid S_t = k \sim \mu_k + \sigma_k \cdot t_\nu$. The Student-t (rather than Gaussian) captures fat tails within each state.

> **4. Poisson Jump Mechanism.** At each time step, with probability $\epsilon$, a "jump event" fires. The duration of the jump $K \sim \text{Poisson}(\lambda)$ determines how many consecutive steps are forced into tail states. With probability $p_{\text{neg}}$, the jump goes to the _bottom_ tail states (crash); otherwise, to the _top_ tail states (rally). This produces the realistic **volatility clustering** seen in real market data — multiple consecutive extreme-return days during crises.

### Fitting Workflow

```
prices → excess_growth_rates → fit(LaplacePartition) → assign_states 
       → estimate_transition → fit_emissions → tune(ε, λ) → JumpHiddenMarkovModel
```

The `tune` step uses a grid search over $(\epsilon, \lambda)$ to match the empirical kurtosis and autocorrelation function of absolute returns — the two key signatures of realistic market behavior that GBM misses.

## Backtesting Framework: Monte Carlo over Regime-Switching Paths

With a fitted JumpHMM, we can generate _hundreds_ of plausible future market paths — each with its own pattern of calm periods, volatility spikes, and crash events. The backtesting framework runs each strategy across all paths and collects the distribution of outcomes:

> **Backtesting Pseudo-code:**
> 1. Fit JumpHMM to training data; tune jump parameters.
> 2. Generate $M$ synthetic market paths of length $T$ (e.g., $M = 200$ paths, $T = 252$ days).
> 3. For each path, generate per-ticker prices via SIM from the market index.
> 4. For each path, run the strategy (engine, buy-and-hold, AI baseline) and record: final wealth, max drawdown, Sharpe ratio.
> 5. Aggregate: compute medians, percentiles, and failure rates across all $M$ paths.

This is _not_ a single backtest on a single path — it's a Monte Carlo simulation that reveals the _distribution_ of possible outcomes. This tells us not just "did the strategy work on one scenario?" but "how often does it fail, and how bad are the failures?"

The key advantage of HMM-generated paths over simple GBM: the jumps produce _realistic_ crash-and-recovery patterns. A strategy that looks good on GBM data but fails on HMM data is not robust enough for deployment.

## The Zero-Level AI Baseline

Before we can claim our rebalancing engine is "AI-powered," we need a baseline that answers: _what if you just asked an AI for advice?_ If a general-purpose language model can match or beat the engine's performance with zero financial engineering, then the engine isn't adding value.

Our "zero-level AI baseline" is a **naive momentum strategy**: rebalance monthly toward recent winners, using a softmax over trailing returns to determine weights. This mimics the kind of advice a general AI might give — "invest more in assets that have been going up recently."

> **Why momentum as the AI baseline?** When asked for portfolio advice, language models typically suggest some variant of: "diversify, but tilt toward recent trends" — which is momentum in disguise. This baseline captures the _spirit_ of generic AI advice without the overhead of actually calling an LLM at each step.

| Strategy | Signal Source | Rebalance Frequency | Risk Controls |
|----------|-------------|-------------------|---------------|
| **AI Engine** (Session 2) | EMA crossover → λ → SIM allocator | Daily (configurable) | Drawdown limits, turnover caps |
| **AI Baseline** (naive) | Trailing return momentum | Monthly | None |
| **Buy-and-Hold** | None | Never | None |

The validation report will show whether the engine's sophistication — sentiment signals, SIM allocation, trigger rules — actually translates into better outcomes than the naive baseline.

## Validation Criteria: Pass/Fail for Deployment

The final output of this session is a **validation report** — a formal document that says whether the rebalancing engine is ready for production (Session 4) or needs more work. The report evaluates each strategy against explicit pass/fail criteria:

| Criterion | Threshold | Rationale |
|-----------|----------|-----------|
| **Median Sharpe ≥ 0.3** | Must beat risk-free on a risk-adjusted basis | If Sharpe < 0.3, the strategy isn't generating enough return per unit of risk |
| **Median Max Drawdown ≤ 25%** | Worst-case loss must be survivable | Drawdowns > 25% trigger investor redemptions and regulatory scrutiny |
| **Failure Rate ≤ 10%** | No more than 10% of paths end below starting capital | A strategy that loses money >10% of the time is unreliable |
| **Engine beats Buy-and-Hold** | Median wealth must exceed passive benchmark | If the engine can't beat passive investing, the complexity isn't justified |

> **What does "pass" mean?** A strategy that passes all four criteria across HMM-generated paths has demonstrated robustness to regime shifts, fat tails, and volatility clustering. It is _not_ guaranteed to work in production — real markets have additional risks (liquidity, execution, model drift) — but it has cleared the minimum bar for deployment consideration.

## Summary

> **Key Takeaways from Session 3:**
> * GBM-generated synthetic data misses volatility clustering, fat tails, and regime persistence — all critical for realistic stress testing
> * JumpHMM combines a Hidden Markov Model with Poisson jumps to produce synthetic paths that match real market statistical signatures
> * Monte Carlo backtesting across hundreds of paths reveals the _distribution_ of outcomes, not just a single best/worst case
> * The "zero-level AI baseline" provides a lower bound — if the engine can't beat naive momentum, the engineering isn't justified
> * Formal validation criteria (Sharpe, drawdown, failure rate) determine whether the engine is ready for production deployment in Session 4

**Next Session:** In [Session 4](../session-4/L4/eCornell-AI-Finance-L4-Lecture-ProductionOps-May-2026.ipynb), strategies that pass validation move to production — with live operating rules, sentiment monitoring, alert systems, and human override protocols.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The models and strategies described are pedagogical tools using synthetic data. Real-world deployment requires additional regulatory, legal, and operational considerations.